# modeling_v8.ipynb — Phase 3(피처 구현) + M0a·M0b·M-T·M1·M2

> **modeling_v8 Phase 3~4** · 실행 위치: `modeling_v8/` 폴더 안 (상대경로 `../문제1(하)`). 커널 **venv (Python 3.12)**. PLAN **v1.5** 반영.
> 원본 `pm_feature/`를 **이식**해 피처 테이블을 만들고(Phase 3), 복원 pkl로 **M0a**(정합, valid≈38.3) → **M0b**(재학습, G1 완결) → **M-T**(시간-only 대조) → **M1**(+C23_te) → **M2**(센서 풀 gain 재선별)까지 수행한다.

## 레짐 피처 (v1.5 — 조용/요란 verdict)
`pm_log.json`은 **verdict 포맷** `[{"date","type","verdict"}]`을 쓴다. **`is_high_regime`**은 최근 PM의 `type`이 아니라 **verdict 상태기계**로 결정한다 — `loud`면 상태 갱신, `quiet`면 유지. **`high_regime_days`**의 앵커는 **"마지막 loud 이벤트"**다(임의 PM 아님) — quiet PM에서 dslp만 리셋되고 hrd는 연속돼 *가짜 신품 스파이크*를 막는다.
> **동치**: 데이터 내 유일 경계(12-24)가 loud이자 major라, verdict 규칙과 옛 type 규칙이 **현 데이터에서 값 동일**(Cell 4 assert). 그래서 M0a~M-T 수치는 전부 불변. 원본 pkl은 옛 이름 `is_post_pm`/`post_pm_days`로 학습됐고 현 데이터에선 `is_high_regime`/`high_regime_days`와 값 동일 → M0a는 옛 이름으로 pkl에 먹인다.

## 게이트
- **G1(a)**: M0a valid ≈ 38.3(±1.5) → 피처 테이블이 원본과 정합함을 증명.
- **G1**: M0b(재학습) valid ≤ 42 & CV ~41 → v8 파이프라인의 CV/valid 확정.
- **M1 채택**: ΔCV(M1−M0b) ≤ −0.3 → C23_te 채택, 아니면 기각.


In [1]:
# ── [Cell 1] 설정 · 상수(config 이식) · pm_log 파서 · pkl 로드 ───────
import json, pickle, warnings
from pathlib import Path
import numpy as np, pandas as pd, lightgbm as lgb
warnings.filterwarnings("ignore")

BASE   = Path("..")
DATA   = BASE / "문제1(하)"
ANS    = BASE / "문제1_하_answer"
PM_LOG = BASE / "pm_log.json"
PKL    = BASE / "pm_feature" / "lgbm_model.pkl"

# 원본 config.py 이식
COLS_ALL_NULL = ['C2','C13','C26','C37','C43','C47','C53','C55']
COLS_CONSTANT = ['C3','C8','C14','C19','C21','C24','C28','C29','C30','C44','C45','C51']
COLS_DUPLICATE= ['C36','C35','C38']
COLS_TO_DROP  = COLS_ALL_NULL + COLS_CONSTANT + COLS_DUPLICATE                 # 23개
FDC_FEATURES  = ['C4','C6','C12','C17','C18','C25','C27','C32','C42','C46','C48','C49',
                 'C50','C52','C54','C56','C57','C58','C59','C60','C61','C62','C63']    # 23종
AGG_FUNCS = ['mean','std','max','min','last']
ID_COL, STEP_COL, TARGET_COL, SORT_COL = 'C64','C7','C65','C40'
FMT = '%Y-%m-%d %H:%M:%S.%f'
REF = pd.Timestamp('2018-12-01')          # 저불량 앵커(=minor 이후, 원본 _REF_DATE)

def parse_pm_log(raw):
    # pm_log 엔트리 → (ts, type, verdict). (v1.5) verdict 없으면 major→loud / 그외→quiet 폴백(옛 포맷 호환).
    out=[]
    for e in raw:
        if isinstance(e, dict):
            ts=pd.Timestamp(e["date"]); ty=e.get("type","major")
            vd=e.get("verdict", "loud" if ty=="major" else "quiet")
        else:
            ts=pd.Timestamp(e); ty="major"; vd="loud"          # 옛 날짜-리스트 → major/loud
        out.append((ts,ty,vd))
    return sorted(out, key=lambda x:x[0])

PM_EVENTS = parse_pm_log(json.loads(PM_LOG.read_text(encoding="utf-8")))
print("pm_events:", [(str(t.date()),ty,vd) for t,ty,vd in PM_EVENTS])

# 복원 pkl 로드 → top-10 피처명·파라미터
booster = pickle.load(open(PKL,"rb"))
PKL_FEATS  = booster.feature_name()
PKL_PARAMS = dict(booster.params)
print("pkl feats(10):", PKL_FEATS)
print("pkl params:", {k:PKL_PARAMS[k] for k in ['learning_rate','num_leaves','min_child_samples'] if k in PKL_PARAMS}, "...")


pm_events: [('2018-12-24', 'major', 'loud')]
pkl feats(10): ['is_post_pm', 'post_pm_days', 'days_since_last_pm', 'C33', 'hour_x_c33', 'dslp_x_hour', 'C60_mean_step4', 'hour', 'C59_mean_step4', 'is_special_recipe']
pkl params: {'learning_rate': 0.029017547696366934, 'num_leaves': 175, 'min_child_samples': 5} ...


In [2]:
# ── [Cell 2] 원본 로드 ────────────────────────────────────────────
train_raw = pd.read_csv(DATA/"train_data.csv")
valid_raw = pd.read_csv(DATA/"valid_X.csv")
test_raw  = pd.read_csv(DATA/"test_X.csv")
print("raw:", train_raw.shape, valid_raw.shape, test_raw.shape)


raw: (123614, 65) (20577, 64) (20510, 64)


In [3]:
# ── [Cell 3] 피처 빌더 (원본 preprocessing + feature_engineering 이식) ──
def preprocess(df):
    df = df.copy()
    df[SORT_COL] = pd.to_datetime(df[SORT_COL], format=FMT)
    df = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns])
    return df.sort_values(SORT_COL).reset_index(drop=True)

def make_fdc_features(df):                       # FDC 23종 × 5통계 × step pivot + C41_max
    numeric = [c for c in FDC_FEATURES if pd.api.types.is_numeric_dtype(df[c])]
    string  = [c for c in FDC_FEATURES if not pd.api.types.is_numeric_dtype(df[c])]
    step_dur = df.groupby([ID_COL,STEP_COL])['C41'].max().rename('C41_max').reset_index()
    agg = df.groupby([ID_COL,STEP_COL])[numeric].agg(AGG_FUNCS)
    agg.columns = ['_'.join(c) for c in agg.columns]
    agg = agg.reset_index().merge(step_dur, on=[ID_COL,STEP_COL])
    wide = agg.pivot(index=ID_COL, columns=STEP_COL)
    wide.columns = [f'{c[0]}_step{int(c[1])}' for c in wide.columns]
    wide = wide.reset_index()
    if string:
        wide = wide.merge(df.groupby(ID_COL)[string].first().reset_index(), on=ID_COL)
    return wide

def _time_feats(dates, pm_events):   # 그룹 A 시간/레짐 (v1.5 verdict) — pm_events=[(ts,type,verdict)] sorted
    dslp, is_post, is_high, hrd, is_high_t, hrd_t = [], [], [], [], [], []
    for d in dates:
        past = [(t,ty,vd) for t,ty,vd in pm_events if t<=d]
        if past:
            last_ts = past[-1][0]; dd = (d-last_ts).total_seconds()/86400
            dslp.append(dd); is_post.append(1)
            state, last_loud = 0, None                       # verdict 상태기계 + last-loud 앵커
            for t,ty,vd in past:
                if vd == "loud": state, last_loud = 1, t     # loud→상태 갱신·앵커 이동, quiet→유지
            is_high.append(state)
            hrd.append((d-last_loud).total_seconds()/86400 if state==1 and last_loud is not None else 0.0)
            lty = past[-1][1]; iht = 1 if lty=="major" else 0   # (옛 type 규칙 — Cell 4 동치 검증용 그림자)
            is_high_t.append(iht); hrd_t.append(dd*iht)
        else:                                                # 관측 PM 이전(=저불량 앵커)
            dslp.append((d-REF).total_seconds()/86400)
            is_post.append(0); is_high.append(0); is_high_t.append(0); hrd.append(0.0); hrd_t.append(0.0)
    return pd.DataFrame({'days_since_last_pm':dslp,'is_post_pm':is_post,'is_high_regime':is_high,
                         'high_regime_days':hrd,'_is_high_type':is_high_t,'_hrd_type':hrd_t}, index=dates.index)

def make_meta_features(df, pm_events):
    wf = df.groupby(ID_COL)
    meta = wf['C33'].first().reset_index()
    wd = wf[SORT_COL].first().reset_index().rename(columns={SORT_COL:'_date'})
    tf = _time_feats(wd['_date'], pm_events)
    wd = pd.concat([wd, tf], axis=1); wd['hour'] = wd['_date'].dt.hour
    cols = [ID_COL,'days_since_last_pm','is_post_pm','is_high_regime','high_regime_days',
            '_is_high_type','_hrd_type','hour']
    meta = meta.merge(wd[cols], on=ID_COL)
    meta['post_pm_days'] = meta['days_since_last_pm'] * meta['is_post_pm']   # pkl 재현용(last-PM 앵커, 옛 이름)
    meta['dslp_x_hour']  = meta['days_since_last_pm'] * meta['hour']
    meta['hour_x_c33']   = meta['hour'] * meta['C33']
    return meta

def build_features(df_raw, pm_events=PM_EVENTS):
    df  = preprocess(df_raw)
    res = make_fdc_features(df).merge(make_meta_features(df, pm_events), on=ID_COL)
    res['is_special_recipe'] = (res['C6']=='C6_1').astype(int)
    res['C23'] = df.groupby(ID_COL)['C23'].first().values     # M1(C23_te) 재료
    if TARGET_COL in df.columns:
        res = res.merge(df.groupby(ID_COL)[TARGET_COL].first().reset_index(), on=ID_COL)
    return res

print("빌더 정의 완료.")


빌더 정의 완료.


In [4]:
# ── [Cell 4] 피처 테이블 생성 + 정합 assert ───────────────────────
Xtr = build_features(train_raw)
Xva = build_features(valid_raw)
Xte = build_features(test_raw)
assert (len(Xtr),len(Xva),len(Xte)) == (11939,1990,1990), "WF 수 불일치"
core10 = ['is_high_regime','high_regime_days','days_since_last_pm','C33','dslp_x_hour',
          'hour','hour_x_c33','C60_mean_step4','C59_mean_step4','is_special_recipe']
miss = [c for c in core10 if c not in Xtr.columns]
assert not miss, f"코어 피처 누락: {miss}"
# v1.5 동치 assert: verdict 규칙 == 옛 type 규칙 (현 데이터 유일 경계 12-24가 loud+major → 동치)
for nm,D in [('train',Xtr),('valid',Xva),('test',Xte)]:
    assert (D['is_high_regime']==D['_is_high_type']).all(),      f"{nm}: is_high_regime(verdict) ≠ is_high(type)"
    assert np.allclose(D['high_regime_days'], D['_hrd_type']),   f"{nm}: high_regime_days(loud앵커) ≠ hrd(type)"
print("✅ v1.5 동치: is_high_regime(verdict)==is_high(type) & high_regime_days(loud앵커)==hrd(type) — 전 세트")
for D in (Xtr,Xva,Xte): D.drop(columns=['_is_high_type','_hrd_type'], inplace=True)   # 그림자 열 제거
print(f"피처 테이블: train {Xtr.shape} / valid {Xva.shape} / test {Xte.shape}")
# pkl 재현 별칭 값-동일: is_high_regime == is_post_pm, high_regime_days == post_pm_days
eq1 = (Xtr['is_high_regime']==Xtr['is_post_pm']).all()
eq2 = np.allclose(Xtr['high_regime_days'], Xtr['post_pm_days'])
print(f"pkl 별칭 값-동일(경계 1회 loud+major): is_high==is_post {eq1}, hrd==post_pm_days {eq2}")
print(f"  급등 {int(Xtr.is_high_regime.sum())} / 저불량 {int((Xtr.is_high_regime==0).sum())} WF")


✅ v1.5 동치: is_high_regime(verdict)==is_high(type) & high_regime_days(loud앵커)==hrd(type) — 전 세트
피처 테이블: train (11939, 569) / valid (1990, 568) / test (1990, 568)
pkl 별칭 값-동일(경계 1회 loud+major): is_high==is_post True, hrd==post_pm_days True
  급등 8247 / 저불량 3692 WF


In [5]:
# ── [Cell 5] M0a — 복원 pkl로 valid/test 직접 예측 (정합 검증) ─────
def rmse_vs_answer(res, ans_path):
    ans = pd.read_csv(ans_path); ans[ID_COL]=ans[ID_COL].astype(str)
    r = res.copy(); r[ID_COL]=r[ID_COL].astype(str)
    pred = booster.predict(r[PKL_FEATS])            # pkl 학습 피처명(is_post_pm 등)으로 예측
    m = r[[ID_COL]].assign(pred=pred).merge(ans.rename(columns={TARGET_COL:'true'}), on=ID_COL)
    return float(np.sqrt(np.mean((m['true']-m['pred'])**2))), len(m)

va_rmse, nva = rmse_vs_answer(Xva, ANS/"valid_Y_answer.csv")
te_rmse, nte = rmse_vs_answer(Xte, ANS/"test_Y_answer.csv")
print(f"[M0a] valid RMSE = {va_rmse:.3f} (n={nva})  |  test RMSE = {te_rmse:.3f} (n={nte})")
G1a = abs(va_rmse-38.3) <= 1.5
print("🟢 G1(a) 통과 — 피처 테이블 정합 (valid≈38.3)" if G1a else "🔴 G1(a) 실패 — diff 디버깅(dslp·pm_date·C59/C60 NaN·피처명)")
print("  → 다음: M0b(동일 10피처·복원 파라미터로 wafer 5-fold 재학습, valid≤42)")


[M0a] valid RMSE = 37.971 (n=1990)  |  test RMSE = 39.041 (n=1990)
🟢 G1(a) 통과 — 피처 테이블 정합 (valid≈38.3)
  → 다음: M0b(동일 10피처·복원 파라미터로 wafer 5-fold 재학습, valid≤42)


In [6]:
# ── [Cell 6] P1 스모크 (v1.5 신세계관 3종) — pm_log 1줄 추가로 자동 대응 + loud앵커 버그 차단 ──
def _build_with(extra):   # pm_log 사본에 가상 이벤트 1줄 추가 → valid 재빌드
    ev = parse_pm_log(json.loads(PM_LOG.read_text(encoding="utf-8")) + [extra])
    return build_features(valid_raw, ev)
base = Xva[[ID_COL,'days_since_last_pm','is_high_regime','high_regime_days']].rename(
        columns={'days_since_last_pm':'dslp0','is_high_regime':'ih0','high_regime_days':'hrd0'})
CUT = 27.3   # 12-24 기준 경과일 27일↑ = 1-20 이후 WF 식별

# (a) 고불량 중 quiet minor: is_high 1유지, hrd 연속(리셋 금지 ← loud앵커), dslp만 리셋
A = _build_with({"date":"2019-01-20","type":"minor","verdict":"quiet"}).merge(base, on=ID_COL)
Aa = A[A.dslp0 > CUT]
a_ok = (Aa.is_high_regime==1).all() and np.allclose(Aa.high_regime_days, Aa.hrd0) and (Aa.days_since_last_pm < Aa.dslp0-0.5).all()
print(f"(a) quiet minor 1-20 → 이후 {len(Aa)}WF: is_high 1유지 & hrd 연속(리셋X) & dslp만 리셋 = {'✅' if a_ok else '❌'}")

# (b) 저불량 중 quiet major: type이 major여도 verdict quiet → 점프 없음(is_high 0유지)
B = _build_with({"date":"2018-12-15","type":"major","verdict":"quiet"}).merge(
      Xva[[ID_COL,'is_high_regime']].rename(columns={'is_high_regime':'ih0'}), on=ID_COL)
Bb = B[B.ih0==0]
b_ok = (Bb.is_high_regime==0).all()
print(f"(b) quiet major 12-15 → 저불량 {len(Bb)}WF: is_high 0유지(quiet=점프 없음) = {'✅' if b_ok else '❌'}")

# (c) 신규 loud major: is_high 갱신, hrd 새 loud 앵커에서 0부터 재시작(hrd==dslp)
C = _build_with({"date":"2019-01-20","type":"major","verdict":"loud"}).merge(base, on=ID_COL)
Cc = C[C.dslp0 > CUT]
c_ok = (Cc.is_high_regime==1).all() and np.allclose(Cc.high_regime_days, Cc.days_since_last_pm)
print(f"(c) loud major 1-20 → 이후 {len(Cc)}WF: is_high 갱신 & hrd 새 loud앵커 재시작(hrd==dslp) = {'✅' if c_ok else '❌'}")

print("  ✅ P1 — pm_log 1줄(date+type+verdict) 추가만으로 전 피처 자동 대응. "
      "(a)가 loud앵커 수정의 '가짜 ~1400 스파이크' 차단을 실증(구 dslp앵커면 hrd=0 오독). 예측-밴드 정밀검증은 M8(§9.4)." if (a_ok and b_ok and c_ok)
      else "  ❌ 스모크 실패 — 빌더 점검")


(a) quiet minor 1-20 → 이후 673WF: is_high 1유지 & hrd 연속(리셋X) & dslp만 리셋 = ✅
(b) quiet major 12-15 → 저불량 606WF: is_high 0유지(quiet=점프 없음) = ✅
(c) loud major 1-20 → 이후 673WF: is_high 갱신 & hrd 새 loud앵커 재시작(hrd==dslp) = ✅
  ✅ P1 — pm_log 1줄(date+type+verdict) 추가만으로 전 피처 자동 대응. (a)가 loud앵커 수정의 '가짜 ~1400 스파이크' 차단을 실증(구 dslp앵커면 hrd=0 오독). 예측-밴드 정밀검증은 M8(§9.4).


## Phase 4 — M0b(재학습) + M-T(시간-only 대조군)

M0a가 "우리 피처 = 원본 피처"를 증명했으니, 이제 **우리 손으로 재학습**해 v8의 CV/valid를 확정한다(**G1 완결**).

- **M0b**: 코어 10피처 + 복원 파라미터로 **wafer 5-fold OOF CV** + full-train(원본 best_iteration 705라운드) → valid/test.
- **M-T**: 시간/레짐 7피처만(센서 3종 `C60/C59_mean_step4`·`is_special_recipe` 제외) 대조군. 이후 모든 마일스톤에서 **"센서 기여 = M-T − 모델"**로 센서 증분을 추적한다(PLAN §8.4 — "시간만 남으면 v5 합칠 의미 없다" 우려 대응).
- **CV 스킴**: 피처 테이블은 **wafer 1행**(고유 C64=행수 11,939) → wafer-level `KFold(5, shuffle, seed42)`. 각 wafer가 단일 그룹이라 이 단계에선 `GroupKFold(C64)`와 목적이 같고, **row-level(M3+)부터** 그룹이 비자명해지므로 그때 `GroupKFold(C64)`로 전환한다.


In [7]:
# ── [Cell 7] M0b — 코어 10피처 재학습 (wafer 5-fold OOF CV + full-train) ──
from sklearn.model_selection import KFold
y = Xtr[TARGET_COL].to_numpy(float)

# 복원 파라미터(pkl) 채택 — Cell 1 PKL_PARAMS에서 학습 하이퍼파라미터
M8_PARAMS = dict(objective='regression', metric='rmse',
    learning_rate   =PKL_PARAMS.get('learning_rate', 0.029017547696366934),
    num_leaves      =PKL_PARAMS.get('num_leaves', 175),
    min_child_samples=PKL_PARAMS.get('min_child_samples', 5),
    feature_fraction=PKL_PARAMS.get('feature_fraction', 0.6324704159196377),
    bagging_fraction=PKL_PARAMS.get('bagging_fraction', 0.864012693783303),
    bagging_freq    =PKL_PARAMS.get('bagging_freq', 7),
    lambda_l1       =PKL_PARAMS.get('lambda_l1', 5.04154328625296),
    lambda_l2       =PKL_PARAMS.get('lambda_l2', 0.024814259264649002),
    min_split_gain  =PKL_PARAMS.get('min_split_gain', 0.2573073648505903),
    verbose=-1, seed=42)
BEST_ROUNDS = 705    # 원본 pkl best_iteration(복원 meta) → full-train 학습 예산

def wafer_cv_oof(feats, params=M8_PARAMS):
    X = Xtr[feats]; oof = np.full(len(y), np.nan)
    for a, b in KFold(5, shuffle=True, random_state=42).split(X):
        d  = lgb.Dataset(X.iloc[a], y[a]); dv = lgb.Dataset(X.iloc[b], y[b], reference=d)
        m  = lgb.train(params, d, num_boost_round=5000, valid_sets=[dv],
                       callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[b] = m.predict(X.iloc[b], num_iteration=m.best_iteration)
    return float(np.sqrt(np.mean((y-oof)**2)))

def full_train(feats, params=M8_PARAMS, rounds=BEST_ROUNDS):
    return lgb.train(params, lgb.Dataset(Xtr[feats], y), num_boost_round=rounds)

def rmse_pred(model, res, feats, ans_path):
    ans = pd.read_csv(ans_path); ans[ID_COL] = ans[ID_COL].astype(str)
    r = res.copy(); r[ID_COL] = r[ID_COL].astype(str)
    p = model.predict(r[feats])
    m = r[[ID_COL]].assign(p=p).merge(ans.rename(columns={TARGET_COL:'t'}), on=ID_COL)
    return float(np.sqrt(np.mean((m['t']-m['p'])**2)))

cv_m0b  = wafer_cv_oof(core10)
mdl_m0b = full_train(core10)
va_m0b  = rmse_pred(mdl_m0b, Xva, core10, ANS/"valid_Y_answer.csv")
te_m0b  = rmse_pred(mdl_m0b, Xte, core10, ANS/"test_Y_answer.csv")
print(f"[M0b] 10피처 재학습   CV(wafer,OOF)={cv_m0b:.3f}  valid={va_m0b:.3f}  test={te_m0b:.3f}")
G1 = (va_m0b <= 42) and (cv_m0b <= 44)
print("🟢 G1 완결 — 재학습 valid≤42 & CV~41 확정" if G1 else "🔴 G1 미완 — 파라미터/피처 점검")
print("  → 다음: M1(+C23_te) — fold 내부 OOF 타깃 인코딩")


[M0b] 10피처 재학습   CV(wafer,OOF)=40.347  valid=38.399  test=38.912
🟢 G1 완결 — 재학습 valid≤42 & CV~41 확정
  → 다음: M1(+C23_te) — fold 내부 OOF 타깃 인코딩


In [8]:
# ── [Cell 8] M-T — 시간-only 대조군 (센서 3종 제외) + 센서 기여 ────
TIME7 = ['is_high_regime','high_regime_days','days_since_last_pm','C33','dslp_x_hour','hour','hour_x_c33']
assert set(core10) - set(TIME7) == {'C60_mean_step4','C59_mean_step4','is_special_recipe'}, "M-T=코어10−센서3 이어야"
cv_mt  = wafer_cv_oof(TIME7)
mdl_mt = full_train(TIME7)
va_mt  = rmse_pred(mdl_mt, Xva, TIME7, ANS/"valid_Y_answer.csv")
te_mt  = rmse_pred(mdl_mt, Xte, TIME7, ANS/"test_Y_answer.csv")
print(f"[M-T] 시간 7피처       CV(wafer,OOF)={cv_mt:.3f}  valid={va_mt:.3f}  test={te_mt:.3f}")
print(f"[센서 기여] CV {cv_mt:.2f}(M-T) − {cv_m0b:.2f}(M0b) = {cv_mt-cv_m0b:+.2f}pt"
      f"  |  valid {va_mt-va_m0b:+.2f}pt  |  test {te_mt-te_m0b:+.2f}pt")
print("  → 센서(C59/C60 집계 + 레시피)가 시간축 위에 유의미하게 +기여 → v5 센서 결합 의미 있음(PLAN §8.4)")


[M-T] 시간 7피처       CV(wafer,OOF)=44.260  valid=44.622  test=44.149
[센서 기여] CV 44.26(M-T) − 40.35(M0b) = +3.91pt  |  valid +6.22pt  |  test +5.24pt
  → 센서(C59/C60 집계 + 레시피)가 시간축 위에 유의미하게 +기여 → v5 센서 결합 의미 있음(PLAN §8.4)


## M1 — +C23_te (그룹 C, 실험으로 채택 판정)

v5의 유일한 실질 발굴이던 **C23 OOF 타깃인코딩**(m=20, fold-내 fit)을 코어 10에 얹어(11피처), **레짐을 통제한 뒤에도 C23에 잔여 신호가 남는지**를 본다. 누수 차단을 위해 **중첩 OOF**로 인코딩한다 — outer-train 행은 내부 5-fold OOF, outer-val 행은 outer-train 전체 통계로 인코딩(전역 fit 금지, PLAN §7.2).

**채택 게이트**: ΔCV = M1 − M0b ≤ **−0.3**이면 채택. E9에서 C23이 시간 클러스터(레짐 프록시)로 판명됐으므로 **기각 예상**(M0가 이미 흡수).


In [9]:
# ── [Cell 9] M1 — +C23_te (11피처, fold-내 중첩 OOF 타깃인코딩 m=20) ──
M_SMOOTH = 20.0
def _fit_te(cat, yv, m=M_SMOOTH):                 # 스무딩 타깃인코딩 fit
    gm = float(np.mean(yv))
    g  = pd.DataFrame({'c':cat.values,'y':yv}).groupby('c')['y'].agg(['sum','count'])
    return (g['sum'] + m*gm) / (g['count'] + m), gm
def _apply_te(cat, enc, gm): return cat.map(enc).fillna(gm).to_numpy(float)

def m1_cv_oof():                                   # outer 5-fold OOF (내부 OOF로 train 인코딩 → 누수 차단)
    X = Xtr[core10]; c23 = Xtr['C23']; oof = np.full(len(y), np.nan)
    for a,b in KFold(5, shuffle=True, random_state=42).split(X):
        te_a = np.full(len(a), np.nan)
        for ia,ib in KFold(5, shuffle=True, random_state=1).split(a):
            enc,gm = _fit_te(c23.iloc[a[ia]], y[a[ia]]); te_a[ib] = _apply_te(c23.iloc[a[ib]], enc, gm)
        enc,gm = _fit_te(c23.iloc[a], y[a]); te_b = _apply_te(c23.iloc[b], enc, gm)   # val=outer-train 전체 통계
        Xa = X.iloc[a].assign(C23_te=te_a); Xb = X.iloc[b].assign(C23_te=te_b)
        d = lgb.Dataset(Xa, y[a]); dv = lgb.Dataset(Xb, y[b], reference=d)
        mm = lgb.train(M8_PARAMS, d, num_boost_round=5000, valid_sets=[dv], callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[b] = mm.predict(Xb, num_iteration=mm.best_iteration)
    return float(np.sqrt(np.mean((y-oof)**2)))

te_full = np.full(len(y), np.nan)                  # full-train용 OOF C23_te
for a,b in KFold(5, shuffle=True, random_state=42).split(Xtr):
    enc,gm = _fit_te(Xtr['C23'].iloc[a], y[a]); te_full[b] = _apply_te(Xtr['C23'].iloc[b], enc, gm)
enc_all,gm_all = _fit_te(Xtr['C23'], y)            # 전체 train → valid/test 인코딩

cv_m1  = m1_cv_oof()
mdl_m1 = lgb.train(M8_PARAMS, lgb.Dataset(Xtr[core10].assign(C23_te=te_full), y), num_boost_round=BEST_ROUNDS)
va_m1  = rmse_pred(mdl_m1, Xva.assign(C23_te=_apply_te(Xva['C23'],enc_all,gm_all)), core10+['C23_te'], ANS/"valid_Y_answer.csv")
te_m1  = rmse_pred(mdl_m1, Xte.assign(C23_te=_apply_te(Xte['C23'],enc_all,gm_all)), core10+['C23_te'], ANS/"test_Y_answer.csv")
dCV = cv_m1 - cv_m0b
print(f"[M1] +C23_te 11피처  CV(wafer,OOF)={cv_m1:.3f}  valid={va_m1:.3f}  test={te_m1:.3f}")
print(f"[채택 게이트] ΔCV = {cv_m1:.3f}(M1) − {cv_m0b:.3f}(M0b) = {dCV:+.3f}  → "
      + ("✅ 채택 (레짐 통제 후에도 C23 잔여 신호)" if dCV<=-0.3 else "❌ 기각 — C23_te는 레짐 프록시(M0가 이미 흡수), 잔여 신호 없음"))
print("  → 다음: M2(센서 집계 풀 gain 재선별, C25 우선)")


[M1] +C23_te 11피처  CV(wafer,OOF)=41.028  valid=38.783  test=39.197
[채택 게이트] ΔCV = 41.028(M1) − 40.347(M0b) = +0.681  → ❌ 기각 — C23_te는 레짐 프록시(M0가 이미 흡수), 잔여 신호 없음
  → 다음: M2(센서 집계 풀 gain 재선별, C25 우선)


## M2 — 센서 집계 풀 투입 → gain 재선별 (그룹 C, 채택 게이트)

코어 10 + **그룹 C 집계 풀 전체**(23센서×5통계×step + C41_max)를 넣고, 1차 학습의 **gain으로 재선별**(2-pass, FS §6)한다. 레짐이 잡힌 상태에서 gain 랭킹이 재편돼 **더 나은 조합**이 나오는지, 특히 EDA에서 급등 내 +0.459였던 **C25**가 부상하는지 본다.

**채택 게이트**: 최고 TOP_N 조합이 M0b 대비 ΔCV ≤ **−0.3** → 채택. (보조로 "코어10 + 신규센서 K개"도 확인.)


In [11]:
# ── [Cell 10] M2 — 센서 집계 풀 투입 → gain 재선별 (2-pass) ──
NON_FEAT = {ID_COL, TARGET_COL, 'C23', 'C6', 'is_post_pm', 'post_pm_days'}   # id·타깃·문자열·pkl별칭 제외
POOL = [c for c in Xtr.columns if c not in NON_FEAT and pd.api.types.is_numeric_dtype(Xtr[c])]
print(f"[M2] 센서 풀 포함 전체 POOL = {len(POOL)}개 (코어10 + 센서 집계 + 시간)")

def cv_oof_gain(feats):                       # wafer 5-fold OOF + fold-평균 gain
    X = Xtr[feats]; oof = np.full(len(y), np.nan); g = np.zeros(len(feats))
    for a, b in KFold(5, shuffle=True, random_state=42).split(X):
        d = lgb.Dataset(X.iloc[a], y[a]); dv = lgb.Dataset(X.iloc[b], y[b], reference=d)
        mm = lgb.train(M8_PARAMS, d, num_boost_round=5000, valid_sets=[dv], callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[b] = mm.predict(X.iloc[b], num_iteration=mm.best_iteration); g += mm.feature_importance('gain')
    return float(np.sqrt(np.mean((y-oof)**2))), dict(zip(feats, g/5))

# pass1 — 전체 풀 CV + gain 랭킹
pool_cv, gain = cv_oof_gain(POOL); ranked = sorted(gain, key=gain.get, reverse=True); tot = sum(gain.values()) or 1
"""print(f"[M2 pass1] 전체 풀 CV={pool_cv:.3f} (풀 통째론 노이즈)  gain TOP10(상대%):")
for i, f in enumerate(ranked[:10], 1):
    tag = ' ★C25' if f.startswith('C25_') else (' ·신센서' if f not in core10 else '')
    print(f"   {i:2d}. {f:20s} {100*gain[f]/tot:5.1f}%{tag}")"""
print(f"[M2 pass1] 전체 풀 CV={pool_cv:.3f} (풀 통째론 노이즈)  gain TOP15(상대%):")
for i, f in enumerate(ranked[:15], 1):
    tag = ' ★C25' if f.startswith('C25_') else (' ·신센서' if f not in core10 else '')
    print(f"   {i:2d}. {f:20s} {100*gain[f]/tot:5.1f}%{tag}")
c25 = [f for f in ranked if f.startswith('C25_')]
print(f"   C25 aggregate {len(c25)}개 · 최고 gain 순위 {ranked.index(c25[0])+1 if c25 else '-'}")

# pass2 — gain TOP_N 재선별 스윕
best = (None, 1e9, None)
for N in [10, 12, 15, 20, 25]:
    cvN = wafer_cv_oof(ranked[:N]); print(f"   TOP_{N:2d}  CV={cvN:.3f}  ΔvsM0b={cvN-cv_m0b:+.3f}")
    if cvN < best[1]: best = (N, cvN, ranked[:N])
N, cvB, topB = best

# 보조 probe — 코어10 유지 + gain 상위 신규 센서 K개 추가
new_ranked = [f for f in ranked if f not in core10]
pb = min((wafer_cv_oof(core10 + new_ranked[:K]), K) for K in [3, 5, 8])
print(f"   [probe] 코어10 + 신규센서 최선(+{pb[1]})  CV={pb[0]:.3f}  Δ={pb[0]-cv_m0b:+.3f}")

mdl_m2 = lgb.train(M8_PARAMS, lgb.Dataset(Xtr[topB], y), num_boost_round=BEST_ROUNDS)
va_m2 = rmse_pred(mdl_m2, Xva, topB, ANS/"valid_Y_answer.csv"); te_m2 = rmse_pred(mdl_m2, Xte, topB, ANS/"test_Y_answer.csv")
dCV = cvB - cv_m0b
print(f"[M2 최고] gain TOP_{N}  CV={cvB:.3f}  valid={va_m2:.3f}  test={te_m2:.3f}")
print(f"[채택 게이트] ΔCV = {cvB:.3f} − {cv_m0b:.3f}(M0b) = {dCV:+.3f}  → "
      + ("✅ 채택" if dCV<=-0.3 else "❌ 기각 — 센서 풀 재선별이 코어 10을 못 넘음"))
print("  → C59/C60_mean_step4는 개별 gain 낮아도(~18위) 집단으로 필수(FS §5): gain 선택이 이를 떨궈 악화. C25도 미부상(23위). 코어 10 유지. 다음 M3(row-level).")


[M2] 센서 풀 포함 전체 POOL = 563개 (코어10 + 센서 집계 + 시간)
[M2 pass1] 전체 풀 CV=53.006 (풀 통째론 노이즈)  gain TOP15(상대%):
    1. is_high_regime        50.2%
    2. high_regime_days      27.2%
    3. days_since_last_pm     9.1%
    4. C12_mean_step6         4.4% ·신센서
    5. C33                    1.7%
    6. dslp_x_hour            0.5%
    7. C12_mean_step7         0.5% ·신센서
    8. hour                   0.4%
    9. C4_mean_step4          0.3% ·신센서
   10. C4_max_step1           0.3% ·신센서
   11. hour_x_c33             0.2%
   12. C12_max_step6          0.2% ·신센서
   13. C61_mean_step1         0.1% ·신센서
   14. C60_std_step4          0.1% ·신센서
   15. C62_mean_step1         0.1% ·신센서
   C25 aggregate 25개 · 최고 gain 순위 23
   TOP_10  CV=47.017  ΔvsM0b=+6.670
   TOP_12  CV=46.996  ΔvsM0b=+6.649
   TOP_15  CV=43.937  ΔvsM0b=+3.590
   TOP_20  CV=45.076  ΔvsM0b=+4.729
   TOP_25  CV=45.836  ΔvsM0b=+5.489
   [probe] 코어10 + 신규센서 최선(+3)  CV=41.052  Δ=+0.705
[M2 최고] gain TOP_15  CV=43.937  valid=41.674  test=43.253
[채택 게

## 다음 단계 (M3~)
- ✅ **M0b/M-T**(G1 완결, 센서 +3.91pt) · ✅ **M1**(C23_te 기각) · ✅ **M2**(센서 풀 재선별 기각 — 코어 10이 이미 최적, C59/C60의 집단 필수성·C25 미부상 확인).
- **M3** row-level 결합(v5 빌더 이식, 기대 하향 — 레짐 피처가 WF 상수라 이득 소멸 예상) · **M4** TOP_N·블록 ablation·시간 dedup 최종 확정(**G2**).
- 이후 **M5**(모델 벤치: xgboost/catboost) · **M7**(Lot-CV/time-split) · **M8**(신규 trace 시뮬레이션 — 스모크 예측-밴드 정밀검증 포함).
